In [1]:
import numpy as np
import matplotlib.pyplot as plt
import sys
import xarray
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)
from sklearn.ensemble import HistGradientBoostingRegressor
sys.path.append("../src/climate_trends/")

import load as ld
import xarray as xr
import pandas as pd
from statsmodels.tsa.seasonal import STL
import emcee
from statsmodels.tsa.statespace.structural import UnobservedComponents


# Summary
In this notebook I apply two classical ML algorithm formalisms to predict the average temperature for the next months.

# Load the data [for Paris]

In [13]:
## Load the dataset for Paris
ds_p=ld.load_single_file("../data/raw/era5_paris_t2m.nc")
t2m_p = ds_p["t2m"].sel(latitude=48.75,longitude=2.25,method='nearest') -273.15
df = pd.DataFrame({
    "valid_time": pd.to_datetime(t2m_p["valid_time"].values),
    "t2m": t2m_p.values
})
## Convert to panda format
print(type(df["t2m"]),type(df["valid_time"]))
test_size = int(len(df)*0.2)
train_size=int(len(df)*0.8)

<class 'pandas.core.series.Series'> <class 'pandas.core.series.Series'>


# Baseline Model - Climatology

In [43]:
## Predict the next month temperature from a monthly average

 # Convert the time coordinate to a pandas datetime index
time_index = pd.to_datetime(t2m_p.valid_time.values)
    
df_c = pd.DataFrame(t2m_p.values, index=time_index, columns=['t2m'])

######### Apply climatology ############################

train = df_c.iloc[:train_size]
test = df_c.iloc[train_size:]
train_index = train.index
test_index = test.index

# --------------------------------------------------
# Compute climatology from TRAIN ONLY
# --------------------------------------------------

monthly_climatology = train.groupby(train.index.month)["t2m"].mean()

monthly_stats = (
    train.groupby(train.index.month)["t2m"]
    .agg(
        median="median",
        lower=lambda x: x.quantile(0.10),
        upper=lambda x: x.quantile(0.80)
    )
)
# --------------------------------------------------
# Predict test period
# --------------------------------------------------
prediction = test.index.month.map(
    monthly_stats["median"]
)
error_lower =prediction - test.index.month.map(
    monthly_stats["lower"]
)

error_upper = -prediction+test.index.month.map(
    monthly_stats["upper"]
)


#prediction = test.index.month.map(monthly_climatology)

#prediction = pd.Series(
 #   prediction.values,
 #   index=test.index
#)

# --------------------------------------------------
# Metrics
# --------------------------------------------------
#prediction=test["climatology"]

mse = mean_squared_error(test, prediction)
rmse = np.sqrt(mse)
mae = mean_absolute_error(test, prediction)
r2= r2_score(test, prediction)


# Store metrics in a DataFrame
metrics_df = pd.DataFrame({
    "Metric": ["RMSE", "MAE", "R2"],
    "Value": [rmse, mae, r2],
    "Unit": ["°C", "°C", "-"]
})
    # Last observation date
last_date = df_c.index[-1]

# Next month number (1-12)
next_month = (last_date.month % 12) + 1

# Climatology forecast
#next_month_prediction = monthly_climatology.loc[next_month]
next_month_prediction = monthly_stats.loc[next_month, "median"]
next_month_lower = next_month_prediction - monthly_stats.loc[next_month, "lower"]
next_month_upper = monthly_stats.loc[next_month, "upper"] - next_month_prediction

print("=" * 40)
print("Evaluation Metrics")
print("=" * 40)
print(f"RMSE : {rmse:.3f} °C")
print(f"MAE  : {mae:.3f} °C")
print(f"R²   : {r2:.3f}")

print("\nForecast for next month")
print(f"{next_month_prediction:.2f} °C")

Evaluation Metrics
RMSE : 2.055 °C
MAE  : 1.722 °C
R²   : 0.868

Forecast for next month
3.63 °C


## Feature Engineering

In [48]:
# ----------------------------
# Time features
# ----------------------------
df["year"] = df["valid_time"].dt.year
df["month"] = df["valid_time"].dt.month

# Long-term trend (months since start)
df["trend"] = np.arange(len(df))

# Seasonal encoding
df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)

# ----------------------------
# Lag Features (Previous 12 months)
# ----------------------------
for lag in range(1, 13):
    df[f"lag_{lag}"] = df["t2m"].shift(lag)


# ------------------------------------------------------------
# Rolling Statistics
# ------------------------------------------------------------

df["rolling_mean_3"] = df["t2m"].shift(1).rolling(3).mean()
df["rolling_mean_6"] = df["t2m"].shift(1).rolling(6).mean()
df["rolling_mean_12"] = df["t2m"].shift(1).rolling(12).mean()

df["rolling_std_3"] = df["t2m"].shift(1).rolling(3).std()
df["rolling_std_6"] = df["t2m"].shift(1).rolling(6).std()
df["rolling_std_12"] = df["t2m"].shift(1).rolling(12).std()



# ----------------------------
# Feature columns
# ----------------------------
#feature_cols = ["trend", "month_cos","month_sin", "rolling_mean_12","rolling_std_12", 
#                "rolling_mean_3","rolling_std_3","rolling_mean_6","rolling_std_6",
#    ] + [f"lag_{lag}" for lag in range(1, 13)]

feature_cols = ["trend", "month_cos","month_sin"
    ] + [f"lag_{lag}" for lag in range(1, 13)]
# Save the last column for forecasting
# Row used for forecasting
forecast_df = df.tail(1).copy()
X_forecast=forecast_df[feature_cols]
# Define the target variable for prediction
df["target"] = df["t2m"].shift(-1)

# ----------------------------
# Remove rows with NAN values
# ----------------------------
df = df.dropna().reset_index(drop=True)

## Define the X and Y of dataset for prediction
X = df[feature_cols]
y = df["target"]

## Test Train Split

In [49]:
X_train = X.iloc[:-test_size]
X_test = X.iloc[-test_size:]

y_train = y.iloc[:-test_size]
y_test = y.iloc[-test_size:]

# Gradient Boost Model

In [52]:
# ------------------------------------------------------------
# Gradient Boosting Model
# ------------------------------------------------------------

model = HistGradientBoostingRegressor(
    learning_rate=0.05,
    max_depth=6,
    max_iter=500,
    random_state=42,
)

model.fit(X_train, y_train)


# ------------------------------------------------------------
# Prediction
# ------------------------------------------------------------

y_pred = model.predict(X_test)


# ------------------------------------------------------------
# Lower prediction bound - 10th percentile
# ------------------------------------------------------------

lower_model = HistGradientBoostingRegressor(
    loss="quantile",
    quantile=0.10,
    learning_rate=0.05,
    max_depth=6,
    max_iter=500,
    random_state=42,
)

lower_model.fit(X_train, y_train)

y_lower = lower_model.predict(X_test)


# ------------------------------------------------------------
# Upper prediction bound - 90th percentile
# ------------------------------------------------------------

upper_model = HistGradientBoostingRegressor(
    loss="quantile",
    quantile=0.90,
    learning_rate=0.05,
    max_depth=6,
    max_iter=500,
    random_state=42,
)

upper_model.fit(X_train, y_train)

y_upper = upper_model.predict(X_test)


# ------------------------------------------------------------
# Evaluation
# ------------------------------------------------------------

rmse = np.sqrt(
    mean_squared_error(y_test, y_pred)
)

mae = mean_absolute_error(
    y_test,
    y_pred
)

r2 = r2_score(
    y_test,
    y_pred
)


# ------------------------------------------------------------
# Prediction interval evaluation
# ------------------------------------------------------------

# Fraction of actual observations inside the
# 10th-90th percentile prediction interval

coverage = np.mean(
    (y_test >= y_lower) &
    (y_test <= y_upper)
)

# Average width of the prediction interval

interval_width = np.mean(
    y_upper - y_lower
)


# ------------------------------------------------------------
# Print results
# ------------------------------------------------------------

print("=" * 40)
print("Evaluation Metrics")
print("=" * 40)

print(f"RMSE : {rmse:.3f} °C")
print(f"MAE  : {mae:.3f} °C")
print(f"R²   : {r2:.3f}")

print("-" * 40)

print("Prediction Interval (10th-90th percentile)")
print(f"Coverage      : {coverage:.3f} ({coverage * 100:.1f}%)")
print(f"Average Width : {interval_width:.3f} °C")

Evaluation Metrics
RMSE : 1.959 °C
MAE  : 1.583 °C
R²   : 0.881
----------------------------------------
Prediction Interval (10th-90th percentile)
Coverage      : 0.456 (45.6%)
Average Width : 2.624 °C


## Get the prediction for Jan,25 (From Gradient Boost)

In [54]:
# ------------------------------------------------------------
# Next month prediction
# ------------------------------------------------------------

next_month_prediction = model.predict(X_forecast)[0]

# Lower and upper prediction bounds
next_month_lower = lower_model.predict(X_forecast)[0]
next_month_upper = upper_model.predict(X_forecast)[0]

# Error bar sizes
lower_error = next_month_prediction - next_month_lower
upper_error = next_month_upper - next_month_prediction


# ------------------------------------------------------------
# Print forecast
# ------------------------------------------------------------

print("\n" + "=" * 40)
print("Forecast for next month")
print("=" * 40)

print(f"Prediction : {next_month_prediction:.2f} °C")
print(
    f"Range      : "
    f"{next_month_lower:.2f} – {next_month_upper:.2f} °C"
)

print(
    f"Error bars : "
    f"-{lower_error:.2f} / +{upper_error:.2f} °C"
)


Forecast for next month
Prediction : 5.28 °C
Range      : 2.64 – 6.34 °C
Error bars : -2.64 / +1.06 °C


# Random Forest Model

In [ ]:
from sklearn.ensemble import RandomForestRegressor

# ------------------------------------------------------------
# Random Forest Model
# ------------------------------------------------------------
model_RF = RandomForestRegressor(
    n_estimators=500,
    max_depth=15,
    min_samples_split=2,
    min_samples_leaf=1,
    max_features='sqrt',
    bootstrap=True,
    random_state=42,
    n_jobs=-1
)

# Train
model_RF.fit(X_train, y_train)

# ------------------------------------------------------------
# Prediction
# ------------------------------------------------------------
y_pred_RF = model_RF.predict(X_test)
# ------------------------------------------------------------
# Evaluation
# ------------------------------------------------------------

rmse = np.sqrt(mean_squared_error(y_test, y_pred_RF))
mae = mean_absolute_error(y_test, y_pred_RF)
r2 = r2_score(y_test, y_pred_RF)

print("=" * 40)
print("Evaluation Metrics")
print("=" * 40)
print(f"RMSE : {rmse:.3f} °C")
print(f"MAE  : {mae:.3f} °C")
print(f"R²   : {r2:.3f}")

Evaluation Metrics
RMSE : 1.777 °C
MAE  : 1.433 °C
R²   : 0.902


In [55]:
from sklearn.ensemble import RandomForestRegressor

# ------------------------------------------------------------
# Random Forest Model
# ------------------------------------------------------------

model_RF = RandomForestRegressor(
    n_estimators=500,
    max_depth=15,
    min_samples_split=2,
    min_samples_leaf=1,
    max_features='sqrt',
    bootstrap=True,
    random_state=42,
    n_jobs=-1
)


# ------------------------------------------------------------
# Train
# ------------------------------------------------------------

model_RF.fit(X_train, y_train)


# ------------------------------------------------------------
# Prediction
# ------------------------------------------------------------

y_pred_RF = model_RF.predict(X_test)


# ------------------------------------------------------------
# Individual tree predictions
# ------------------------------------------------------------

tree_predictions = np.array([
    tree.predict(X_test)
    for tree in model_RF.estimators_
])

# Shape:
# (500 trees, number of test observations)


# ------------------------------------------------------------
# Prediction bounds
# ------------------------------------------------------------

y_lower_RF = np.percentile(
    tree_predictions,
    10,
    axis=0
)

y_upper_RF = np.percentile(
    tree_predictions,
    90,
    axis=0
)


# ------------------------------------------------------------
# Error bar sizes
# ------------------------------------------------------------

error_lower_RF = (
    y_pred_RF - y_lower_RF
)

error_upper_RF = (
    y_upper_RF - y_pred_RF
)


# ------------------------------------------------------------
# Evaluation
# ------------------------------------------------------------

rmse = np.sqrt(
    mean_squared_error(
        y_test,
        y_pred_RF
    )
)

mae = mean_absolute_error(
    y_test,
    y_pred_RF
)

r2 = r2_score(
    y_test,
    y_pred_RF
)


# ------------------------------------------------------------
# Prediction interval coverage
# ------------------------------------------------------------

coverage_RF = np.mean(
    (y_test >= y_lower_RF) &
    (y_test <= y_upper_RF)
)

interval_width_RF = np.mean(
    y_upper_RF - y_lower_RF
)


# ------------------------------------------------------------
# Print results
# ------------------------------------------------------------

print("=" * 40)
print("Random Forest Evaluation Metrics")
print("=" * 40)

print(f"RMSE : {rmse:.3f} °C")
print(f"MAE  : {mae:.3f} °C")
print(f"R²   : {r2:.3f}")

print("-" * 40)

print("Prediction Interval (10th-90th percentile)")
print(
    f"Coverage      : "
    f"{coverage_RF:.3f} "
    f"({coverage_RF * 100:.1f}%)"
)

print(
    f"Average Width : "
    f"{interval_width_RF:.3f} °C"
)

/home/dell/miniconda3/lib/python3.14/site-packages/sklearn/utils/validation.py:2820: UserWarning: X has feature names, but DecisionTreeRegressor was fitted without feature names
  warnings.warn(
/home/dell/miniconda3/lib/python3.14/site-packages/sklearn/utils/validation.py:2820: UserWarning: X has feature names, but DecisionTreeRegressor was fitted without feature names
  warnings.warn(
/home/dell/miniconda3/lib/python3.14/site-packages/sklearn/utils/validation.py:2820: UserWarning: X has feature names, but DecisionTreeRegressor was fitted without feature names
  warnings.warn(
/home/dell/miniconda3/lib/python3.14/site-packages/sklearn/utils/validation.py:2820: UserWarning: X has feature names, but DecisionTreeRegressor was fitted without feature names
  warnings.warn(
/home/dell/miniconda3/lib/python3.14/site-packages/sklearn/utils/validation.py:2820: UserWarning: X has feature names, but DecisionTreeRegressor was fitted without feature names
  warnings.warn(
/home/dell/miniconda3/lib

Random Forest Evaluation Metrics
RMSE : 1.834 °C
MAE  : 1.495 °C
R²   : 0.895
----------------------------------------
Prediction Interval (10th-90th percentile)
Coverage      : 0.769 (76.9%)
Average Width : 4.650 °C


## Get the prediction for Jan,25 (From Random Forest)

In [56]:
# ------------------------------------------------------------
# Next month Random Forest prediction
# ------------------------------------------------------------

next_month_prediction = model_RF.predict(X_forecast)[0]


# ------------------------------------------------------------
# Individual tree predictions
# ------------------------------------------------------------

next_month_tree_predictions = np.array([
    tree.predict(X_forecast.values)[0]
    for tree in model_RF.estimators_
])


# ------------------------------------------------------------
# Lower and upper prediction bounds
# ------------------------------------------------------------

next_month_lower = np.percentile(
    next_month_tree_predictions,
    10
)

next_month_upper = np.percentile(
    next_month_tree_predictions,
    90
)


# ------------------------------------------------------------
# Error bar sizes
# ------------------------------------------------------------

lower_error = (
    next_month_prediction - next_month_lower
)

upper_error = (
    next_month_upper - next_month_prediction
)


# ------------------------------------------------------------
# Print forecast
# ------------------------------------------------------------

print("\n" + "=" * 40)
print("Random Forest Forecast for Next Month")
print("=" * 40)

print(
    f"Prediction : "
    f"{next_month_prediction:.2f} °C"
)

print(
    f"Range      : "
    f"{next_month_lower:.2f} – "
    f"{next_month_upper:.2f} °C"
)

print(
    f"Error bars : "
    f"-{lower_error:.2f} / "
    f"+{upper_error:.2f} °C"
)


Random Forest Forecast for Next Month
Prediction : 4.04 °C
Range      : 0.52 – 6.65 °C
Error bars : -3.52 / +2.61 °C


# SARIMA for prediction

In [ ]:

from statsmodels.tsa.statespace.sarimax import SARIMAX
# --------------------------------------------------------
# Load dataset
# --------------------------------------------------------

df = pd.DataFrame({
    "valid_time": pd.to_datetime(t2m_p["valid_time"].values),
    "t2m": t2m_p.values
})

df = df.set_index("valid_time")

y = df["t2m"].copy()


# --------------------------------------------------------
# Train / Test split
# --------------------------------------------------------

train_size = int(len(y) * 0.8)

train = y.iloc[:train_size]
test = y.iloc[train_size:]


# --------------------------------------------------------
# SARIMA Model
# --------------------------------------------------------

model = SARIMAX(
    train,
    order=(2, 1, 2),
    seasonal_order=(1, 1, 1, 12),
    enforce_stationarity=False,
    enforce_invertibility=False,
)

results = model.fit()


# --------------------------------------------------------
# Predict test period
# --------------------------------------------------------

pred = results.get_forecast(
    steps=len(test)
)

pred_mean = pred.predicted_mean


# --------------------------------------------------------
# Prediction interval
# --------------------------------------------------------

# alpha = 0.20 gives an 80% prediction interval

conf_int = pred.conf_int(
    alpha=0.20
)

lower = conf_int.iloc[:, 0]
upper = conf_int.iloc[:, 1]


# --------------------------------------------------------
# Error bars
# --------------------------------------------------------

error_lower = pred_mean - lower
error_upper = upper - pred_mean


# --------------------------------------------------------
# Metrics
# --------------------------------------------------------

mse = mean_squared_error(
    test,
    pred_mean
)

rmse = np.sqrt(mse)

mae = mean_absolute_error(
    test,
    pred_mean
)

r2 = r2_score(
    test,
    pred_mean
)


# --------------------------------------------------------
# Prediction interval metrics
# --------------------------------------------------------

coverage = np.mean(
    (test >= lower) &
    (test <= upper)
)

interval_width = np.mean(
    upper - lower
)


# --------------------------------------------------------
# Store metrics in DataFrame
# --------------------------------------------------------

metrics_df = pd.DataFrame({

    "Metric": [
        "RMSE",
        "MAE",
        "R2",
        "Interval_Coverage",
        "Average_Interval_Width"
    ],

    "Value": [
        rmse,
        mae,
        r2,
        coverage,
        interval_width
    ],

    "Unit": [
        "°C",
        "°C",
        "-",
        "%",
        "°C"
    ]
})


# --------------------------------------------------------
# Save metrics
# --------------------------------------------------------



# --------------------------------------------------------
# Save test predictions + error bars
# --------------------------------------------------------

prediction_df = pd.DataFrame({

    "date": test.index,

    "actual": test.values,

    "prediction": pred_mean.values,

    "lower": lower.values,

    "upper": upper.values,

    "error_lower": error_lower.values,

    "error_upper": error_upper.values
})





# --------------------------------------------------------
# Forecast next month
# --------------------------------------------------------

final_model = SARIMAX(
    y,
    order=(2, 1, 2),
    seasonal_order=(1, 1, 1, 12),
    enforce_stationarity=False,
    enforce_invertibility=False,
)

final_results = final_model.fit()


forecast = final_results.get_forecast(
    steps=1
)


# --------------------------------------------------------
# Next month prediction
# --------------------------------------------------------

next_month_prediction = (
    forecast.predicted_mean.iloc[0]
)


# --------------------------------------------------------
# Next month prediction interval
# --------------------------------------------------------

next_month_conf_int = forecast.conf_int(
    alpha=0.20
)

next_month_lower = (
    next_month_conf_int.iloc[0, 0]
)

next_month_upper = (
    next_month_conf_int.iloc[0, 1]
)


# --------------------------------------------------------
# Next month error bars
# --------------------------------------------------------

next_month_error_lower = (
    next_month_prediction -
    next_month_lower
)

next_month_error_upper = (
    next_month_upper -
    next_month_prediction
)


# --------------------------------------------------------
# Print next month forecast
# --------------------------------------------------------

print("\n" + "=" * 40)
print("SARIMA Forecast for Next Month")
print("=" * 40)

print(
    f"Prediction : "
    f"{next_month_prediction:.2f} °C"
)

print(
    f"Range      : "
    f"{next_month_lower:.2f} – "
    f"{next_month_upper:.2f} °C"
)

print(
    f"Error bars : "
    f"-{next_month_error_lower:.2f} / "
    f"+{next_month_error_upper:.2f} °C"
)


# --------------------------------------------------------
# Save next month forecast
# --------------------------------------------------------

next_month_df = pd.DataFrame({

    "prediction": [
        next_month_prediction
    ],

    "lower": [
        next_month_lower
    ],

    "upper": [
        next_month_upper
    ],

    "error_lower": [
        next_month_error_lower
    ],

    "error_upper": [
        next_month_error_upper
    ]
})




print(
    "Next month forecast saved to "
    "SARIMA_next_month.csv"
)


# --------------------------------------------------------
# Plot
# --------------------------------------------------------


plt.figure(figsize=(12, 5))

plt.plot(
    train.index,
    train,
    label="Training"
)

plt.plot(
    test.index,
    test,
    label="Observed",
    color="black"
)

plt.plot(
    test.index,
    pred_mean,
    label="SARIMA prediction"
)

plt.fill_between(
    test.index,
    lower,
    upper,
    alpha=0.25,
    label="80% prediction interval"
)

plt.legend()

plt.grid(True)

plt.ylabel(
    "Temperature (°C)"
)

plt.title(
    "SARIMA One-Step Forecast"
)



plt.close()


# --------------------------------------------------------
# Return
# --------------------------------------------------------



/home/dell/miniconda3/lib/python3.14/site-packages/statsmodels/tsa/base/tsa_model.py:480: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
/home/dell/miniconda3/lib/python3.14/site-packages/statsmodels/tsa/base/tsa_model.py:480: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
/home/dell/miniconda3/lib/python3.14/site-packages/statsmodels/tsa/statespace/mlemodel.py:737: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  mlefit = super().fit(
/home/dell/miniconda3/lib/python3.14/site-packages/statsmodels/tsa/base/tsa_model.py:480: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
/home/dell/miniconda3/lib/python3.14/site-packages/statsmodels/tsa/base/tsa_model.py:480: ValueWarning: No frequency information was provided, so inf


SARIMA Forecast for Next Month
Prediction : 5.31 °C
Range      : 3.29 – 7.32 °C
Error bars : -2.01 / +2.01 °C
Next month forecast saved to SARIMA_next_month.csv


### print SARIMA Metrics

In [63]:
metrics_df

,Metric,Value,Unit
0,RMSE,1.552778,°C
1,MAE,1.239049,°C
2,R2,0.924658,-
3,Interval_Coverage,0.841530,%
4,Average_Interval_Width,4.285177,°C
